In [ ]:
import os
import json
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/Qwen3_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_45/TiJian_45.json"        
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_45/results_with_qwen3.json"   

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

STOP_TOKENS = ["<|im_end|>", "<|im_start|>", "<|endoftext|>"]

# 1. Load model
print("Loading model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX, n_batch=N_BATCH, n_ubatch=N_UBATCH, n_threads=N_THREADS,
    n_gpu_layers=GPU_LAYERS, main_gpu=MAIN_GPU,
    chat_format="chatml", flash_attn=True, f16_kv=True, 
    use_mmap=True, use_mlock=False, offload_kqv=True, verbose=False,
)
print("Model loaded.")

# 2. Read data
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Total records: {len(data)}")

# 3. Batch processing
results = []
for idx, item in enumerate(data):
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")
    
    # Construct messages: system contains instruction, user contains examination indicators
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": input_text}
    ]
    
    print(f"\nProcessing {idx+1}/{len(data)} ...")
    
    try:
        # Call model generation (non-streaming, get full response directly)
        response = llm.create_chat_completion(
            messages=messages,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            stream=False,   
            stop=STOP_TOKENS,
        )
        # Extract generated text
        generated_text = response["choices"][0]["message"]["content"]
        
        # Optional: filter out think tag content (if the model outputs it)
        # But normally non-reasoning models don't output this, keep it for simple cleanup if needed
        # Here we keep it as is, or process as needed
        
    except Exception as e:
        print(f"Error on item {idx+1}: {e}")
        generated_text = ""   # Leave empty on error
    
    # Construct new record (keep original fields, add output_Qwen3)
    new_item = item.copy()
    new_item["output_Qwen3"] = generated_text
    results.append(new_item)
    
    # Save checkpoint every 10 records to prevent data loss
    if (idx + 1) % 10 == 0:
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"Checkpoint saved at {idx+1} records.")

# 4. Final save
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nAll done! Results saved to {OUTPUT_JSON}")

Loading model...


llama_context: n_ctx_seq (32768) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.
Total records: 45

Processing 1/45 ...

Processing 2/45 ...

Processing 3/45 ...

Processing 4/45 ...

Processing 5/45 ...

Processing 6/45 ...

Processing 7/45 ...

Processing 8/45 ...

Processing 9/45 ...

Processing 10/45 ...
Checkpoint saved at 10 records.

Processing 11/45 ...

Processing 12/45 ...

Processing 13/45 ...

Processing 14/45 ...

Processing 15/45 ...

Processing 16/45 ...

Processing 17/45 ...

Processing 18/45 ...

Processing 19/45 ...

Processing 20/45 ...
Checkpoint saved at 20 records.

Processing 21/45 ...

Processing 22/45 ...

Processing 23/45 ...

Processing 24/45 ...

Processing 25/45 ...

Processing 26/45 ...

Processing 27/45 ...

Processing 28/45 ...

Processing 29/45 ...

Processing 30/45 ...
Checkpoint saved at 30 records.

Processing 31/45 ...

Processing 32/45 ...

Processing 33/45 ...

Processing 34/45 ...

Processing 35/45 ...

Processing 36/45 ...

Processing 37/45 ...

Processing 38/45 ...

Processing 39/45 ...

Processing 40/45 ...
C